In [ ]:
# ═══════════════════════════════════════════════
# VOLTIQ — LightGBM Load Forecasting
# Fixed & Production Ready
# ═══════════════════════════════════════════════

# CELL 1 — Setup
!pip install -q kaggle lightgbm scikit-learn joblib

from google.colab import files
files.upload()
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d kyleahmurphy/uk-electrical-load
!unzip -q uk-electrical-load.zip -d data

import os
os.makedirs('models', exist_ok=True)

# CELL 2 — Load + Preprocess
import pandas as pd
import numpy as np
import lightgbm as lgb
import joblib
from sklearn.metrics import mean_absolute_error, roc_auc_score

df = pd.read_csv("data/House_1.csv")
df = df[['Time', 'Aggregate']]
df.rename(columns={'Time': 'datetime', 'Aggregate': 'kw'}, inplace=True)

df['kw'] = df['kw'] / 1000.0
df['datetime'] = pd.to_datetime(df['datetime'])
df = df.sort_values('datetime').set_index('datetime')

# Remove partial first day
df = df[df.index.date > df.index.date.min()]

# 15-min resample
df = df.resample('15min').mean()
df['kw'] = df['kw'].interpolate(method='linear')

# Outlier clip
upper = df['kw'].quantile(0.99)
df['kw'] = np.clip(df['kw'], 0, upper)

print(f"Data shape: {df.shape}")
print(f"Date range: {df.index.min()} → {df.index.max()}")
print(f"kW range: {df['kw'].min():.3f} → {df['kw'].max():.3f}")

# CELL 3 — Feature Engineering
# Cyclic time encoding (fixes 23→0 discontinuity)
df['hour_sin'] = np.sin(2 * np.pi * df.index.hour / 24)
df['hour_cos'] = np.cos(2 * np.pi * df.index.hour / 24)
df['dow_sin']  = np.sin(2 * np.pi * df.index.dayofweek / 7)
df['dow_cos']  = np.cos(2 * np.pi * df.index.dayofweek / 7)

# Lag features
df['lag_1']   = df['kw'].shift(1)    # 15 min ago
df['lag_4']   = df['kw'].shift(4)    # 1 hour ago
df['lag_96']  = df['kw'].shift(96)   # yesterday same time
df['lag_672'] = df['kw'].shift(672)  # last week same time

# Rolling features
df['roll_4']  = df['kw'].rolling(4).mean()   # 1 hr smooth
df['roll_24'] = df['kw'].rolling(24).mean()  # 6 hr trend

df = df.dropna()

FEATURE_COLS = [
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    'lag_1', 'lag_4', 'lag_96', 'lag_672',
    'roll_4', 'roll_24'
]

print(f"After feature eng: {df.shape}")
print(f"Features: {FEATURE_COLS}")

# CELL 4 — Peak label (do this BEFORE split)
peak_threshold = df['kw'].quantile(0.85)
df['peak'] = (df['kw'] > peak_threshold).astype(int)
print(f"Peak threshold: {peak_threshold:.3f} kW")
print(f"Peak rate: {df['peak'].mean()*100:.1f}%")

# CELL 5 — Train/Val/Test Split (time-based)
n  = len(df)
t1 = int(n * 0.70)
t2 = int(n * 0.85)

X = df[FEATURE_COLS]
y_reg  = df['kw']
y_peak = df['peak']

X_train = X.iloc[:t1]
X_val   = X.iloc[t1:t2]
X_test  = X.iloc[t2:]

y_train = y_reg.iloc[:t1]
y_val   = y_reg.iloc[t1:t2]
y_test  = y_reg.iloc[t2:]

yp_train = y_peak.iloc[:t1]
yp_val   = y_peak.iloc[t1:t2]
yp_test  = y_peak.iloc[t2:]

print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

# CELL 6 — Model 1: Baseline Regressor
print("\nTraining Baseline Model...")

model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    early_stopping_rounds=20,
    verbose=-1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
)

y_pred = model.predict(X_test)

# Metrics
mae  = mean_absolute_error(y_test, y_pred)
mape = np.mean(np.abs(y_pred - y_test) / np.maximum(y_test, 0.1)) * 100
acc_15 = np.mean(
    np.abs(y_pred - y_test) <= 0.15 * np.maximum(y_test, 0.1)
) * 100

print(f"\n{'='*45}")
print(f"  BASELINE MODEL RESULTS")
print(f"{'='*45}")
print(f"  MAE:             {mae:.4f} kW")
print(f"  MAPE:            {mape:.1f}%")
print(f"  Within 15%:      {acc_15:.1f}%  ← quote this")
print(f"{'='*45}")

# CELL 7 — Model 2: P90 Quantile
print("\nTraining P90 Model...")

model_p90 = lgb.LGBMRegressor(
    objective='quantile',
    alpha=0.90,
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    verbose=-1
)

model_p90.fit(X_train, y_train)
p90_pred = model_p90.predict(X_test)

# Validate P90 coverage (critical check)
coverage = np.mean(y_test.values <= p90_pred) * 100
print(f"P90 Coverage: {coverage:.1f}%  (should be ~90%)")
# If <85% → P90 is underestimating → increase alpha to 0.92

# CELL 8 — Model 3: Peak Classifier
print("\nTraining Peak Classifier...")

clf = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    class_weight='balanced',
    verbose=-1
)

clf.fit(
    X_train, yp_train,
    eval_set=[(X_val, yp_val)],
)

peak_prob = clf.predict_proba(X_test)[:, 1]
auc = roc_auc_score(yp_test, peak_prob)

print(f"\n{'='*45}")
print(f"  PEAK CLASSIFIER RESULTS")
print(f"{'='*45}")
print(f"  AUC-ROC:   {auc:.3f}  ← quote this")
print(f"{'='*45}")

# CELL 9 — MILP Output Format
def build_milp_payload(X_input):
    """
    FastAPI /ml/forecast/24hr endpoint mein ye use hoga
    Input: last 672 readings ke features
    Output: MILP-ready JSON
    """
    baseline  = model.predict(X_input)
    p90       = model_p90.predict(X_input)
    peak_p    = clf.predict_proba(X_input)[:, 1]

    # Hourly aggregation (96 slots → 24 hours)
    result = []
    for h in range(24):
        slots = slice(h*4, (h+1)*4)
        result.append({
            "hour":        h,
            "baseline_kw": round(float(np.mean(baseline[slots])), 3),
            "p90_kw":      round(float(np.mean(p90[slots])), 3),
            "peak_prob":   round(float(np.mean(peak_p[slots])), 3),
            # MILP uses this:
            "dynamic_cap": round(float(max(0.5, 3.5 - np.mean(baseline[slots]))), 3),
            "risk_buffer": round(float(np.mean(peak_p[slots]) * 0.5), 3)
        })

    return result

# Test it
sample_payload = build_milp_payload(X_test.values[:96])
print("\nSample MILP payload (first 3 hours):")
for item in sample_payload[:3]:
    print(f"  Hour {item['hour']:2d}: baseline={item['baseline_kw']} | "
          f"p90={item['p90_kw']} | peak_prob={item['peak_prob']} | "
          f"cap={item['dynamic_cap']}")

# CELL 10 — Save everything
joblib.dump(model,      'models/baseline.pkl')
joblib.dump(model_p90,  'models/p90.pkl')
joblib.dump(clf,        'models/peak.pkl')
joblib.dump(FEATURE_COLS, 'models/feature_cols.pkl')

import json
accuracy_info = {
    "mae_kw":       round(float(mae), 4),
    "mape_pct":     round(float(mape), 1),
    "within_15pct": round(float(acc_15), 1),
    "peak_auc":     round(float(auc), 3),
    "p90_coverage": round(float(coverage), 1),
    "model_type":   "LightGBM_3model",
    "features":     FEATURE_COLS
}

with open('models/accuracy_info.json', 'w') as f:
    json.dump(accuracy_info, f, indent=2)

print("\n✅ All models saved!")
print(json.dumps(accuracy_info, indent=2))

# CELL 11 — Download
from google.colab import files
files.download('models/baseline.pkl')
files.download('models/p90.pkl')
files.download('models/peak.pkl')
files.download('models/feature_cols.pkl')
files.download('models/accuracy_info.json')

print("""
Copy to voltiq-backend/models/:
  baseline.pkl
  p90.pkl
  peak.pkl
  feature_cols.pkl
  accuracy_info.json
""")

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/kyleahmurphy/uk-electrical-load
License(s): CC0-1.0
100% 886M/886M [00:59<00:00, 15.7MB/s]

Data shape: (61296, 1)
Date range: 2013-10-10 00:00:00 → 2015-07-10 11:45:00
kW range: 0.130 → 2.466
After feature eng: (60624, 11)
Features: ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'lag_1', 'lag_4', 'lag_96', 'lag_672', 'roll_4', 'roll_24']
Peak threshold: 0.912 kW
Peak rate: 15.0%
Train: 42,436 | Val: 9,094 | Test: 9,094

Training Baseline Model...

  BASELINE MODEL RESULTS
  MAE:             0.1024 kW
  MAPE:            27.1%
  Within 15%:      56.5%  ← quote this

Training P90 Model...
P90 Coverage: 89.9%  (should be ~90%)

Training Peak Classifier...

  PEAK CLASSIFIER RESULTS
  AUC-ROC:   0.972  ← quote this

Sample MILP payload (first 3 hours):
  Hour  0: baseline=0.332 | p90=0.372 | peak_prob=0.0 | cap=3.168
  Hour  1: baseline=0.325 | p90=0.359 | peak_prob=0.0 | cap=3.175
  Hour  2: baseline=0.314 | p9

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



✅ All models saved!
{
  "mae_kw": 0.1024,
  "mape_pct": 27.1,
  "within_15pct": 56.5,
  "peak_auc": 0.972,
  "p90_coverage": 89.9,
  "model_type": "LightGBM_3model",
  "features": [
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "lag_1",
    "lag_4",
    "lag_96",
    "lag_672",
    "roll_4",
    "roll_24"
  ]
}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Copy to voltiq-backend/models/:
  baseline.pkl
  p90.pkl
  peak.pkl
  feature_cols.pkl
  accuracy_info.json

